# SnowFrame Notebook Demo

SnowFrame lets you register Pandas DataFrames as SQL tables, run
warehouse-style SQL using DuckDB, and pull results back into Pandas —
all without leaving your notebook.

**Install (once):**
```bash
pip install -e ".[notebook]"
jupyter lab
```

## 1 · Load the extension

In [ ]:
%load_ext snowframe

## 2 · Create sample DataFrames

In [ ]:
import pandas as pd
from snowframe import SnowFrame, set_active_session

sales = pd.DataFrame({
    "customer_id": [1, 1, 2, 3, 3],
    "amount":      [100, 2500, 700, 1800, 300],
})

customers = pd.DataFrame({
    "customer_id": [1, 2, 3],
    "name":        ["Amit", "Riya", "Kabir"],
})

## 3 · Create a session and auto-register DataFrames

In [ ]:
sf = SnowFrame()
sf.auto()              # scans local scope and registers 'sales' and 'customers'
set_active_session(sf) # makes sf available to %%sf / %%snowframe cells

sf.tables()

## 4 · Create a table with SQL magic

DDL statements execute silently and print the updated table list.

In [ ]:
%%sf

CREATE OR REPLACE TABLE final AS
SELECT
    c.name,
    SUM(s.amount) AS total_amount
FROM sales s
JOIN customers c
    ON s.customer_id = c.customer_id
GROUP BY c.name
ORDER BY total_amount DESC;

## 5 · Pull a table back into Pandas

In [ ]:
final_df = sf.to_df("final")
final_df

## 6 · Query with SQL magic

SELECT statements display the result DataFrame inline.

In [ ]:
%%sf
SELECT * FROM final ORDER BY total_amount DESC;

## 7 · Convenience methods

In [ ]:
# Preview — first N rows (default 10)
sf.show("final")

In [ ]:
# Schema — column names and types
sf.describe("final")

## 8 · SHOW TABLES via magic

In [ ]:
%%sf
SHOW TABLES;

## 9 · Export results

In [ ]:
sf.to_csv("final", "/tmp/final.csv")
print("Exported to /tmp/final.csv")